In [ ]:
# LangGraph + Agentic Tools Demo Notebook# ======================================# This notebook demonstrates how to build a simple agentic workflow using LangGraph.# The workflow includes:#   1. Fetching stock prices from Alpha Vantage API#   2. Fetching current weather from Weatherstack API#   3. A fallback LLM chat (Google Gemini)## Prerequisites:# - You must have a `.env` file with the following keys:#       WEATHERSTACK_API_KEY=your_weatherstack_key#       ALPHAVANTAGE_API_KEY=your_alphavantage_key#       GOOGLE_API_KEY=your_google_api_key# ------------------------------# 1. Imports and Environment Setup# ------------------------------import osimport requestsfrom dotenv import load_dotenv# LangChain / LangGraph importsfrom langchain_google_genai import ChatGoogleGenerativeAIfrom langchain_core.tools import toolfrom langgraph.graph import StateGraph, ENDfrom langgraph.prebuilt import ToolNode# Load environment variables from `.env`load_dotenv()# ------------------------------# 2. Define Tools (Weather, Stock Price, Simple Chat)# ------------------------------@tooldef get_weather(city: str) -> str:    """    Fetches the current weather for a given city using Weatherstack API.    Args:        city (str): Name of the city    Returns:        str: Weather description    """    api_key = os.getenv("WEATHERSTACK_API_KEY")    url = f"http://api.weatherstack.com/current?access_key={api_key}&query={city}"    response = requests.get(url)    data = response.json()    if "current" in data:        temp = data["current"]["temperature"]        desc = data["current"]["weather_descriptions"][0]        return f"The weather in {city} is {desc} with temperature {temp}°C."    else:        return "Sorry, could not fetch weather data."@tooldef get_stock_price(symbol: str) -> str:    """    Fetches the latest stock price for a given ticker symbol using Alpha Vantage API.    Args:        symbol (str): Stock ticker (e.g., AAPL, MSFT)    Returns:        str: Latest stock price    """    api_key = os.getenv("ALPHAVANTAGE_API_KEY")    url = f"https://www.alphavantage.co/query?function=GLOBAL_QUOTE&symbol={symbol}&apikey={api_key}"    response = requests.get(url)    data = response.json()    if "Global Quote" in data and "05. price" in data["Global Quote"]:        price = data["Global Quote"]["05. price"]        return f"The current stock price of {symbol} is ${price}."    else:        return "Sorry, could not fetch stock price."# ------------------------------# 3. LLM Setup (Google Gemini)# ------------------------------llm = ChatGoogleGenerativeAI(    model="gemini-1.5-flash",  # lightweight Gemini model    google_api_key=os.getenv("GOOGLE_API_KEY"))# ------------------------------# 4. Define Graph State# ------------------------------from typing import TypedDictclass GraphState(TypedDict):    input: str   # User input    output: str  # Final response# ------------------------------# 5. Define Nodes# ------------------------------# LLM Node for fallback chatdef llm_chat_node(state: GraphState):    """    Handles general chat queries using Gemini LLM.    """    user_input = state["input"]    response = llm.invoke(user_input)    return {"output": response.content}# ToolNode wrapper for weather and stock toolstool_node = ToolNode([get_weather, get_stock_price])# ------------------------------# 6. Router Function (Decide which node to use)# ------------------------------def route_input(state: GraphState):    """    Routes user query to the correct tool or LLM.    """    query = state["input"].lower()    if "weather" in query:        return "tools"    elif "stock" in query or "price" in query:        return "tools"    else:        return "llm"# ------------------------------# 7. Build Graph# ------------------------------# Create state graphworkflow = StateGraph(GraphState)# Add nodesworkflow.add_node("llm", llm_chat_node)workflow.add_node("tools", tool_node)# Add edges with routingworkflow.set_conditional_entry_point(    route_input,    {        "tools": "tools",        "llm": "llm"    })# End edgesworkflow.add_edge("tools", END)workflow.add_edge("llm", END)# Compile graphapp = workflow.compile()# ------------------------------# 8. Run Examples# ------------------------------# Example 1: Weather Queryresult1 = app.invoke({"input": "What is the weather in London?"})print("\n[Weather Query Result]\n", result1["output"])# Example 2: Stock Price Queryresult2 = app.invoke({"input": "Tell me the stock price of AAPL"})print("\n[Stock Query Result]\n", result2["output"])# Example 3: General Chatresult3 = app.invoke({"input": "Who is the CEO of Google?"})print("\n[Chat Query Result]\n", result3["output"])